# Evaluation 2: Video-LLaVA-7B on RDMB-100

This notebook evaluates Video-LLaVA-7B on 100 physical RGB-D clips using eight uniformly sampled keyframes per clip.

In [ ]:
import os
import re
import glob
import subprocess

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch

subprocess.run(
    "pip install -q transformers==4.49.0 accelerate>=0.26.0 torchvision av",
    shell=True,
    check=True
)

In [ ]:
WORK_DIR = "/kaggle/working/dataset"
os.makedirs(WORK_DIR, exist_ok=True)

tar_candidates = glob.glob(
    "/kaggle/input/**/motion_blindness_100clips.tar.gz", recursive=True
) + glob.glob("/kaggle/input/**/*.tar.gz", recursive=True)

if tar_candidates:
    subprocess.run(
        f"tar -xzf '{tar_candidates[0]}' -C {WORK_DIR}",
        shell=True,
        check=True
    )

In [ ]:
csv_candidates = glob.glob(
    f"{WORK_DIR}/**/master_annotations_100clips.csv", recursive=True
) + glob.glob(
    "/kaggle/input/**/master_annotations_100clips.csv", recursive=True
)

frame_dirs = glob.glob(
    f"{WORK_DIR}/**/verify_frames", recursive=True
) + glob.glob(
    "/kaggle/input/**/verify_frames", recursive=True
)

CSV_PATH = csv_candidates[0]
FRAME_DIR = frame_dirs[0]

subprocess.run(
    f"mv '{FRAME_DIR}/cilp_20' '{FRAME_DIR}/clip_020' 2>/dev/null || true",
    shell=True
)

In [ ]:
def load_clean_clip_frames(clip_id, n_frames=8, target_size=(224, 224)):
    clip_dir = os.path.join(FRAME_DIR, clip_id)

    if not os.path.exists(clip_dir):
        for alias in ["clip_020", "clip_20", "cilp_20"]:
            alt_dir = os.path.join(FRAME_DIR, alias)
            if os.path.exists(alt_dir):
                clip_dir = alt_dir
                break

    files = sorted(glob.glob(os.path.join(clip_dir, "*_rgb.jpg")))
    if not files:
        files = sorted(glob.glob(os.path.join(clip_dir, "*.jpg")))

    if not files:
        return []

    valid_files = files[1:] if len(files) >= 3 else files

    if len(valid_files) <= n_frames:
        selected_files = valid_files
    else:
        step = (len(valid_files) - 1) / (n_frames - 1)
        selected_files = [
            valid_files[int(round(i * step))]
            for i in range(n_frames)
        ]

    images = [
        Image.open(file_path).convert("RGB").resize(target_size)
        for file_path in selected_files
    ]

    return images

In [ ]:
raw_df = pd.read_csv(CSV_PATH)

norm_df = pd.DataFrame()
norm_df["clip_id"] = raw_df[
    "Clip ID" if "Clip ID" in raw_df.columns else "clip_id"
].astype(str).str.strip().replace("cilp_20", "clip_020")

norm_df["scenario"] = raw_df[
    "Scenario" if "Scenario" in raw_df.columns else "scenario"
]

norm_df["depth_motion"] = raw_df[
    "Depth Motion" if "Depth Motion" in raw_df.columns else "depth_motion"
]

for q in range(1, 5):
    text_column = f"Q{q} Text" if f"Q{q} Text" in raw_df.columns else f"q{q}_text"
    correct_column = f"Q{q} Correct" if f"Q{q} Correct" in raw_df.columns else f"q{q}_correct"

    norm_df[f"q{q}_text"] = raw_df[text_column]
    norm_df[f"q{q}_correct"] = raw_df[correct_column].astype(str).str.strip().str.upper()

norm_df.head()

In [ ]:
from transformers import VideoLlavaForConditionalGeneration, VideoLlavaProcessor

MODEL_ID = "LanguageBind/Video-LLaVA-7B-hf"
OUT_CSV = "/kaggle/working/results_video_llava_7b.csv"

processor = VideoLlavaProcessor.from_pretrained(MODEL_ID)

model = VideoLlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
results = []

for idx, row in tqdm(norm_df.iterrows(), total=len(norm_df), desc="Video-LLaVA Benchmark"):
    images = load_clean_clip_frames(
        row["clip_id"],
        n_frames=8,
        target_size=(224, 224)
    )

    if not images:
        continue

    video_tensor = np.stack([np.array(image) for image in images])

    record = {
        "clip_id": row["clip_id"],
        "scenario": row["scenario"],
        "depth_motion": row["depth_motion"]
    }

    for q in range(1, 5):
        prompt = (
            "USER: <video>\n"
            f"Question:\n{row[f'q{q}_text']}\n\n"
            "Answer strictly with ONLY the single letter choice "
            "(A, B, C, or D).\nASSISTANT:"
        )

        inputs = processor(
            text=prompt,
            videos=video_tensor,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=4,
                do_sample=False
            )

            output_text = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )[0].strip().upper()

        if "ASSISTANT:" in output_text:
            output_text = output_text.split("ASSISTANT:")[-1].strip()

        match = re.search(r"\b([A-D])\b", output_text)
        prediction = match.group(1) if match else "INVALID"

        record[f"q{q}_pred"] = prediction
        record[f"q{q}_correct"] = row[f"q{q}_correct"]
        record[f"q{q}_acc"] = prediction == row[f"q{q}_correct"]

    results.append(record)

    if (idx + 1) % 10 == 0:
        torch.cuda.empty_cache()

df_vllava = pd.DataFrame(results)
df_vllava.to_csv(OUT_CSV, index=False)

In [ ]:
print("Video-LLaVA evaluation complete.")

for q in range(1, 5):
    accuracy = df_vllava[f"q{q}_acc"].mean() * 100
    print(f"Category {q}: {accuracy:.1f}%")